# Spark Assignment – Retail Transactions
Business Intelligence and Data Engineering, MSc in Business Analytics

In this notebook I analyse about 1 million online retail transactions using PySpark.
Part 1 answers the Spark SQL questions and Part 2 clusters the customers with KMeans.

## Setup
First I create the Spark session. It runs locally on my laptop using all the CPU cores.
(Spark needs Java 17 or 21 installed.)

In [1]:
# install pyspark and pandas only if they are missing
# (pandas is only used to show the results as nice tables; pandas 3 doesn't work with pyspark 4.0 yet)
import importlib.util, subprocess, sys
for module, package in [("pyspark", "pyspark"), ("pandas", "pandas<3")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, LongType, IntegerType,
                               DoubleType, StringType, BooleanType, TimestampType)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("retail-assignment")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")  # less noise in the output
pd.set_option("display.max_columns", None)


def to_table(df, n=20):
    """Shows the first n rows of a Spark dataframe as a pandas table.
    Dates are turned into text first, because toPandas() can crash on them (timezone problem)."""
    for name, dtype in df.dtypes:
        if dtype == "timestamp":
            df = df.withColumn(name, F.date_format(name, "yyyy-MM-dd HH:mm:ss"))
    return df.limit(n).toPandas()


spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/01 20:02:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.0.1'

## Loading the data
I give the schema myself instead of using `inferSchema`, so Spark doesn't need an extra pass over the file
and the `Date` column is read as a timestamp. Change `DATA_PATH` if the csv is somewhere else.

In [2]:
import os, zipfile

DATA_PATH = "retail-dataset.csv"   # put the csv in the same folder as the notebook

# the dataset was given as a zip, so if only the zip is there I extract the csv from it
if not os.path.exists(DATA_PATH) and os.path.exists(DATA_PATH + ".zip"):
    with zipfile.ZipFile(DATA_PATH + ".zip") as z:
        csv_name = [n for n in z.namelist() if n.endswith(".csv") and "__MACOSX" not in n][0]
        DATA_PATH = z.extract(csv_name, ".")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Put retail-dataset.csv (or the .zip) next to this notebook or change DATA_PATH")

schema = StructType([
    StructField("Transaction_ID", LongType()),
    StructField("Date", TimestampType()),
    StructField("Customer_Name", StringType()),
    StructField("Product", StringType()), # it's a list but stored as text
    StructField("Total_Items", IntegerType()),
    StructField("Total_Cost", DoubleType()),
    StructField("Payment_Method", StringType()),
    StructField("City", StringType()),
    StructField("Store_Type", StringType()),
    StructField("Discount_Applied", BooleanType()),
    StructField("Customer_Category", StringType()),
    StructField("Season", StringType()),
    StructField("Promotion", StringType()),
])

retail = spark.read.csv(DATA_PATH, header=True, schema=schema)
retail.printSchema()

root
 |-- Transaction_ID: long (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: boolean (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)



I cache the dataframe because I will query it many times below.

In [3]:
retail.cache()
print("rows:", retail.count())
to_table(retail, 5)

[Stage 0:=============================>                             (4 + 4) / 8]

rows: 1000000


,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000000,2022-01-21 06:27:29,Stacey Price,"['Ketchup', 'Shaving Cream', 'Light Bulbs']",3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,None
1,1000000001,2023-03-01 13:01:21,Michelle Carlson,"['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'P...",2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One)
2,1000000002,2024-03-21 15:37:04,Lisa Graves,['Spinach'],6,41.49,Credit Card,Houston,Department Store,True,Professional,Winter,None
3,1000000003,2020-10-31 09:59:47,Mrs. Patricia May,"['Tissues', 'Mustard']",1,39.34,Mobile Payment,Chicago,Pharmacy,True,Homemaker,Spring,None
4,1000000004,2020-12-10 00:59:59,Susan Mitchell,['Dish Soap'],10,16.42,Debit Card,Houston,Specialty Store,False,Young Adult,Winter,Discount on Selected Items


## Part 1 – Spark SQL

### Q1. Descriptive statistics for each column
`describe()` gives count, mean, stddev, min and max for every column. It only works with numbers and text,
so Spark skips `Date` and `Discount_Applied` here and I look at those two separately below.

In [4]:
to_table(retail.describe())

,summary,Transaction_ID,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Customer_Category,Season,Promotion
0,count,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000
1,mean,1.0004999995E9,None,None,5.495941,52.45522039999989,None,None,None,None,None,None
2,stddev,288675.2789323441,None,None,2.871654187209307,27.41698914533324,None,None,None,None,None,None
3,min,1000000000,Aaron Acevedo,"['Air Freshener', 'Air Freshener', 'Air Freshe...",1,5.0,Cash,Atlanta,Convenience Store,Homemaker,Fall,BOGO (Buy One Get One)
4,max,1000999999,Zoe York,['Yogurt'],10,100.0,Mobile Payment,Seattle,Warehouse Club,Young Adult,Winter,None


For the two numeric columns I also use `summary()`, which adds the quartiles (25%, 50%, 75%):

In [5]:
to_table(retail.select("Total_Items", "Total_Cost").summary())

,summary,Total_Items,Total_Cost
0,count,1000000,1000000
1,mean,5.495941,52.45522039999989
2,stddev,2.871654187209307,27.41698914533324
3,min,1,5.0
4,25%,3,28.71
5,50%,5,52.42
6,75%,8,76.18
7,max,10,100.0


The date range and the true/false split of `Discount_Applied`:

In [6]:
display(to_table(retail.select(F.min("Date").alias("first_date"), F.max("Date").alias("last_date"))))
display(to_table(retail.groupBy("Discount_Applied").count()))

,first_date,last_date
0,2020-01-01 00:03:54,2024-05-18 19:31:03


,Discount_Applied,count
0,True,500104
1,False,499896


Finally, how many distinct values and how many missing (null) values each column has:

In [7]:
# distinct values per column
display(to_table(retail.select([F.countDistinct(c).alias(c) for c in retail.columns])))

# missing values per column
display(to_table(retail.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in retail.columns])))

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000,996336,329738,571947,10,9501,4,10,6,2,8,4,3


,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,0,0,0,0,0,0,0,0,0,0,0,0,0


**Comments.** There are no missing values in any column. The data covers 1,000,000 transactions from January 2020
to May 2024, in 10 cities, 6 store types, 8 customer categories and 4 payment methods. Each purchase has 1–10 items
and costs between 5 and 100 (about 52 on average). The mean and stddev of `Transaction_ID` don't mean anything since
it is just an id. Also, in `Promotion` "no promotion" is written as the text `None` and not as a real null,
which I need to remember for Q5.

### Q2. Customers that pay in cash
Part 1 is about Spark SQL, so from here on I write the queries in SQL. First I register the dataframe as a temporary view called `retail`, so `spark.sql()` can use it like a normal table.

In [8]:
retail.createOrReplaceTempView("retail")

#I create a new df "cash" form my SQL query
cash = spark.sql("""
    SELECT Customer_Name, Payment_Method
    FROM retail
    WHERE Payment_Method = 'Cash'
""")

print("cash purchases:", cash.count())
print("different customers:", cash.select("Customer_Name").distinct().count())
to_table(cash, 10) #I limit the table to 10 because there are ~140.000 unique customers that payed with cash

cash purchases: 250230
different customers: 140516


,Customer_Name,Payment_Method
0,Michelle Carlson,Cash
1,Joshua Frazier,Cash
2,Victoria Garrett,Cash
3,Angela Jones,Cash
4,Don Harris,Cash
5,Donald Bradford,Cash
6,Nicole Hernandez,Cash
7,Paige White,Cash
8,Mr. Lance Perkins,Cash
9,Paula Maldonado,Cash


**Comments.** 250,230 purchases (about 1 in 4) were paid in cash, which makes sense since there are 4 payment methods. They were made by 140,516 different customers, so many customers appear more than once. I only show the first 10 rows here, because the full result is too long to print.

### Q3. Purchases made by students or teenagers
The `Customer_Category` column has the values `Student` and `Teenager`, so I keep the rows with one of these two using `IN` (same as writing `= 'Student' OR = 'Teenager'`). I keep all the columns, since the question asks for the purchases.

In [9]:
young = spark.sql("""
    SELECT *
    FROM retail
    WHERE Customer_Category IN ('Student', 'Teenager')
""")

print("purchases by students or teenagers:", young.count())
to_table(young, 10)

purchases by students or teenagers: 250161


,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000006,2023-01-08 10:40:03,Victoria Garrett,"['Honey', 'BBQ Sauce', 'Soda', 'Olive Oil', 'G...",4,5.28,Cash,Boston,Specialty Store,False,Student,Summer,Discount on Selected Items
1,1000000012,2023-05-27 15:52:59,Sheila Mcgee,"['Salmon', 'Shaving Cream']",9,39.75,Debit Card,New York,Pharmacy,False,Student,Summer,Discount on Selected Items
2,1000000020,2021-10-31 07:46:22,Angela Jones,"['Pancake Mix', 'Vacuum Cleaner']",7,83.86,Cash,Houston,Pharmacy,False,Teenager,Winter,Discount on Selected Items
3,1000000024,2023-02-27 16:46:56,Hannah Rowland,"['Sponges', 'Vacuum Cleaner', 'Iron']",4,24.03,Debit Card,Dallas,Pharmacy,False,Student,Winter,None
4,1000000028,2022-08-13 01:26:19,Clayton Roth,"['Tomatoes', 'Salmon']",7,21.08,Credit Card,San Francisco,Specialty Store,True,Teenager,Fall,None
5,1000000029,2020-02-27 10:23:56,Dennis Alvarez,"['Tea', 'Salmon', 'Bath Towels', 'Toothpaste']",7,75.38,Mobile Payment,Houston,Convenience Store,True,Teenager,Summer,Discount on Selected Items
6,1000000031,2024-05-04 13:22:31,Kristin Webb,"['Extension Cords', 'Razors', 'Chips', 'Shampoo']",4,13.34,Debit Card,Houston,Department Store,False,Teenager,Summer,None
7,1000000034,2023-02-07 14:09:36,Stacy Owen,"['Yogurt', 'Ice Cream', 'Insect Repellent']",5,34.27,Credit Card,San Francisco,Supermarket,True,Teenager,Summer,BOGO (Buy One Get One)
8,1000000038,2020-07-11 22:15:24,Kimberly Baldwin,"['Shower Gel', 'Dish Soap', 'Plant Fertilizer'...",7,61.55,Credit Card,New York,Convenience Store,False,Student,Fall,None
9,1000000043,2020-10-12 12:39:54,Paige White,"['Dustpan', 'BBQ Sauce', 'Toothbrush', 'Deodor...",8,29.82,Cash,Atlanta,Specialty Store,False,Student,Winter,Discount on Selected Items


**Comments.** 250,161 purchases (about 25%) were made by students or teenagers. This fits with the 8 customer categories being almost equally big (~125,000 purchases each), so two of them together give about a quarter of the data.

### Q4. Customer categories and number of purchases per category
I group the rows by `Customer_Category` and count the rows in each group. I sort from the biggest to the smallest category so the result is easier to read.

In [10]:
categories = spark.sql("""
    SELECT Customer_Category, COUNT(*) AS purchases
    FROM retail
    GROUP BY Customer_Category
    ORDER BY purchases DESC
""")

to_table(categories)

,Customer_Category,purchases
0,Senior Citizen,125485
1,Homemaker,125418
2,Teenager,125319
3,Retiree,125072
4,Student,124842
5,Professional,124651
6,Middle-Aged,124636
7,Young Adult,124577


**Comments.** There are 8 customer categories and they are almost the same size: from 124,577 (Young Adult) to 125,485 (Senior Citizen) purchases, so each one is about 12.5% of the data. No group of customers buys clearly more often than the others, which shows that the dataset is synthetic.

### Q5. The 10 most recent purchases with a promotion
`Promotion` has 3 values: `Discount on Selected Items`, `BOGO (Buy One Get One)` and `None`. As I saw in Q1, `None` is written as text and is not a real null, so I can't use `IS NOT NULL` here. I remove it with `<> 'None'`, sort by date from newest to oldest and keep the first 10.

In [ ]:
recent_promo = spark.sql("""
    SELECT *
    FROM retail
    WHERE Promotion <> 'None'
    ORDER BY Date DESC
    LIMIT 10
""")

to_table(recent_promo)

**Comments.** All 10 purchases are from 18 May 2024, which is the last day in the data, between 18:53 and 19:29. About two thirds of all purchases (666,057) have a promotion. Something strange I noticed: all these purchases are in May, but `Season` says Summer or Fall for some of them, and some have a promotion but `Discount_Applied` is False. This is another sign that the data is synthetic (randomly generated).